# Upload a local JSONL corpus to Hugging Face

Helper for future sessions: validate a JSONL file, attach a dataset card, push
it, then verify that it loads back.

Requires `HF_TOKEN` in the environment (never hardcode it).


In [ ]:
from huggingface_hub import HfApi
from datasets import load_dataset
import json, os
from pathlib import Path

api = HfApi()
REPO_PREFIX = "michaelowusuntim6"
LOCAL_JSONL = "datasets/formatted/code_review/train.jsonl"
REPO_NAME = "my-new-corpus-qwen35"
LICENSE = "apache-2.0"   # must be a valid HF identifier: apache-2.0, mit, cc-by-4.0, other


In [ ]:
# Validate: every line is JSON with a messages list of >= 2 entries, and every
# message content must be a STRING. A list of typed objects (the vision format)
# breaks FastLanguageModel, so it is rejected here.
VALID_ROLES = {"system", "user", "assistant"}
path = Path(LOCAL_JSONL)
rows = bad = 0
problems = {}
with path.open(encoding="utf-8") as fh:
    for i, line in enumerate(fh, 1):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            problems["malformed_json"] = problems.get("malformed_json", 0) + 1
            bad += 1
            continue
        rows += 1
        messages = rec.get("messages")
        if not isinstance(messages, list) or len(messages) < 2:
            problems["bad_messages"] = problems.get("bad_messages", 0) + 1; bad += 1; continue
        roles = set()
        for m in messages:
            if not isinstance(m, dict):
                problems["message_not_dict"] = problems.get("message_not_dict", 0) + 1; bad += 1; break
            if m.get("role") not in VALID_ROLES:
                problems["role_invalid"] = problems.get("role_invalid", 0) + 1; bad += 1; break
            roles.add(m["role"])
            content = m.get("content")
            if isinstance(content, list):
                problems["content_not_str"] = problems.get("content_not_str", 0) + 1; bad += 1; break
            if not isinstance(content, str) or not content.strip():
                problems["content_empty_or_wrong"] = problems.get("content_empty_or_wrong", 0) + 1; bad += 1; break
        else:
            if "user" not in roles or "assistant" not in roles:
                problems["missing_user_or_assistant"] = problems.get("missing_user_or_assistant", 0) + 1
                bad += 1
print(f"{path}: {rows} records, {bad} bad")
print("problems:", problems or "none")
assert bad == 0, "fix the data before uploading"


In [ ]:
card = f'''---
language:
  - en
license: {LICENSE}
task_categories:
  - text-generation
tags:
  - qwen3.5
  - conversation
  - instruction-tuning
---
# {REPO_NAME}

Canonical Qwen3.5 chat records, consumed via
`tokenizer.apply_chat_template()`. Special tokens are added by the template,
never embedded in content.
'''
(Path(LOCAL_JSONL).parent / "README.md").write_text(card, encoding="utf-8")
print(card)


In [ ]:
token = os.environ["HF_TOKEN"]   # never hardcode
repo_id = f"{REPO_PREFIX}/{REPO_NAME}"
api.create_repo(repo_id=repo_id, repo_type="dataset", private=False, exist_ok=True)
api.upload_folder(repo_id=repo_id, folder_path=str(Path(LOCAL_JSONL).parent),
                  repo_type="dataset", commit_message=f"upload {REPO_NAME}")
print("https://huggingface.co/datasets/" + repo_id)


In [ ]:
ds = load_dataset(repo_id, split="train")
print(len(ds), "records")
assert "messages" in ds[0] and isinstance(ds[0]["messages"], list)
print(type(ds[0]["messages"][1]["content"]).__name__, "content OK")
print(ds[0]["messages"][1]["content"][:200])
